In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
# COMMAND ----------
 
# Load environment configuration
 
# MAGIC %run "../00_common/01_environment_config"
 
 
# COMMAND ----------
 
# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================
 
bronze_path = f"{s3_root_path}/Bronze/Nisarga/races"
silver_path = f"{s3_root_path}/Silver/Nisarga/races"
 
print("Bronze Path :", bronze_path)
print("Silver Path :", silver_path)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 1 - READ BRONZE RACES DATA
# ============================================================
 
races_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)
 
display(races_df)
 
 
# COMMAND ----------
 
from pyspark.sql import functions as F
 
 
# COMMAND ----------
 
# ============================================================
# STEP 2 - SELECT REQUIRED COLUMNS
# ============================================================
# URL is excluded because it is not required for analytics.
 
races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)
 
display(races_selected_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 3 - STANDARDIZE COLUMN NAMES
# ============================================================
 
races_renamed_df = (
    races_selected_df
    .withColumnsRenamed({
        "raceName": "race_name",
        "circuitId": "circuit_id"
    })
)
 
display(races_renamed_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 4 - REMOVE NULL BUSINESS KEYS
# ============================================================
# Season and round together identify a race.
 
races_valid_df = (
    races_renamed_df
    .filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull()
    )
)
 
display(races_valid_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 5 - REMOVE DUPLICATES
# ============================================================
 
races_distinct_df = (
    races_valid_df
    .dropDuplicates(["season", "round"])
)
 
display(races_distinct_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 6 - STANDARDIZE TEXT VALUES
# ============================================================
 
races_clean_df = (
    races_distinct_df
    .withColumn(
        "race_name",
        F.initcap(F.col("race_name"))
    )
)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 7 - CREATE DERIVED COLUMN
# ============================================================
# race_year is derived from the race date.
 
races_final_df = (
    races_clean_df
    .withColumn(
        "race_year",
        F.year(F.col("date"))
    )
)
 
display(races_final_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 8 - WRITE TO SILVER IN DELTA FORMAT
# ============================================================
 
(
    races_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 9 - VALIDATE SILVER DATA
# ============================================================
 
silver_races_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)
 
display(silver_races_df)
 
print("Bronze Records :", races_df.count())
print("Silver Records :", silver_races_df.count())
 
print("Silver Races data successfully written to:")